# 5.1 章节介绍

## 概述

在第3章，我们学习了如何使用 Vector 进行向量计算；在第4章，我们学习了如何使用 Cube 计算矩阵乘。本章要把这两部分计算接起来。

模型中的一层往往不只做矩阵乘。例如，矩阵乘结束后，还需要给结果加上偏置。

本章先以“矩阵乘 → 加偏置”为例，学习**如何让Cube和Vector在同一个算子中协同工作**，再让它支持动态行数。其他向量后处理作为章末选做内容。

<small>学习准备：本节约10分钟。建议先学完第3章的向量计算和第4章的分块矩阵乘。</small>

### 学习目标

读完本节，你应能：

- 说明本章会用到前两章的哪些知识。
- 指出本章案例中 Cube 和 Vector 各自负责的计算。
- 了解本章各节要解决的问题。

## 1. 把前两章学过的计算接起来

先回顾前两章的学习重点：

- **第3章：** 用Vector完成向量计算，本章用它给矩阵乘结果加偏置。
- **第4章：** 用Cube完成分块矩阵乘，本章在此基础上接上向量后处理。

假设模型中的一层依次完成以下计算：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">计算步骤</th>
<th style="text-align: left; vertical-align: top;">要做什么</th>
<th style="text-align: left; vertical-align: top;">本例主要使用的计算单元</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">矩阵乘（Matmul）</td>
<td style="text-align: left; vertical-align: top;">将输入与权重相乘，得到中间矩阵 X</td>
<td style="text-align: left; vertical-align: top;">Cube，简称 C 核</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">加偏置（Bias）</td>
<td style="text-align: left; vertical-align: top;">给X的每列加上该列对应的偏置值b，得到Y</td>
<td style="text-align: left; vertical-align: top;">Vector，简称V核</td>
</tr>
</tbody>
</table>

<small>表中列出的是两步计算的分工。数据搬运、结果传递和写回将在 5.2 节结合代码介绍。</small>

例如，X 的某一行为 [-1, 0, 2]，偏置为 [0.5, 1, -0.5]。两者相加后得到Y的这一行[-0.5, 1, 1.5]。X的其他行也使用同一组偏置。

**关键点：** 得到矩阵乘结果 `X` 后，还需要加上偏置，才能得到最终输出 `Y`。

## 2. 本章的融合算子是什么意思

本章讨论的是**矩阵乘与向量后处理的融合**。我们把矩阵乘和偏置相加放在同一个算子中：C 核计算矩阵乘，V 核处理矩阵乘结果，最后输出 Y。

对于同一个结果块，偏置相加必须等矩阵乘结果就绪后才能开始。因此，编写融合算子时，除了写出两部分计算，还要安排好中间结果的传递。

融合为性能优化提供了两类机会：

- **减少中间结果的搬运。** 如果平台和实现允许直接把矩阵乘结果交给 Vector，就有机会省去中间结果写入 GM、再从 GM 读出的过程。
- **让 Cube 和 Vector 重叠工作。** 在缓冲和同步安排合适的情况下，Vector 给第一个结果块加偏置时，Cube 可以同时计算下一个独立结果块的矩阵乘。不同结果块的计算在时间上重叠，这就是 C/V 流水的一种组织方式，目的是减少计算单元的等待。

**融合并不自动保证这些收益。** 能省去哪些搬运、能否形成有效的计算重叠，取决于具体实现；最终是否提速，还需要测量。

5.2 先结合代码讲清楚一个结果块的计算和传递过程，帮助你写出正确的融合算子，再在节末说明进一步组织 C/V 流水的优化方向。


## 3. 为什么从 Matmul + Bias 开始

我们选择 Matmul + Bias 作为第一个融合案例，主要有三个原因：

1. **矩阵乘已经学过，可以在已有代码上继续。** 第4章的分块矩阵乘负责前半部分计算，本章可以集中讲解怎样接上 Vector 后处理。
2. **偏置相加足够简单，便于看清新增代码。** 每个结果元素只需加上对应列的偏置。数学计算容易理解，学习时就能把注意力放在 DSL 语句、数据位置和硬件操作之间的对应关系上。
3. **案例虽小，计算和传递过程完整。** 它包含 Cube 计算、矩阵乘结果传给 Vector、Vector 加偏置，以及最终结果写回，能够完整展示本章要学习的 C/V 协同过程。

5.2 将沿着这条数据路径逐段解释代码，说明每一步使用什么数据、执行什么操作，以及为什么这样写。


## 4. 输入数量变化时：以动态行数 M 为例

以模型中一个固定的线性层为例，每一行输入表示一个样本或一个 token 的特征。输入有 M 行，每行有 K 个特征；这一层将每行转换为 N 个输出特征。

在这个场景中，K 和 N 由该层权重矩阵的形状确定，保持不变；每次送入的样本数或 token 数可能不同，因此 **M 会随本次输入数量变化**。我们希望同一个编译好的算子能够处理这些输入，而不必为每一种行数重新编译。

5.3 以动态 M 为例介绍动态形状适配：在代码中声明运行时行数，根据实际行数确定任务数量，并在调用前检查输入。本例固定 N、K，只支持 M 为64的正整数倍，具体原因会在5.3说明。

选择 M 是为了贴合这一应用场景，并集中讲清一个维度变化时需要做的修改。其他场景中 N、K 也可能变化；动态尺寸的声明和检查思路可以借鉴，但相应的索引、循环和边界处理需要另行实现与验证。

## 本章学习安排


<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">小节</th>
<th style="text-align: left; vertical-align: top;">主要内容</th>
<th style="text-align: left; vertical-align: top;">学完后应能完成什么</th>
<th style="text-align: left; vertical-align: top;">建议时间</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="05.01_introduction.ipynb">5.1 章节介绍</a></td>
<td style="text-align: left; vertical-align: top;">了解本章与 Vector、Cube 计算的关系</td>
<td style="text-align: left; vertical-align: top;">解释融合案例中 C 核和 V 核的分工</td>
<td style="text-align: left; vertical-align: top;">10分钟</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="05.02_matmul_bias.ipynb">5.2 融合案例</a></td>
<td style="text-align: left; vertical-align: top;">实现Matmul + Bias</td>
<td style="text-align: left; vertical-align: top;">逐行读懂DSL，并对应到硬件上的计算和搬运</td>
<td style="text-align: left; vertical-align: top;">45分钟</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="05.03_dynamic_shapes.ipynb">5.3 动态形状</a></td>
<td style="text-align: left; vertical-align: top;">处理输入行数的变化</td>
<td style="text-align: left; vertical-align: top;">编写动态版本，并检查输入是否符合要求</td>
<td style="text-align: left; vertical-align: top;">35分钟</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="05.04_practice.ipynb">5.4 章节实践</a></td>
<td style="text-align: left; vertical-align: top;">补全融合代码、组合动态M、验证结果</td>
<td style="text-align: left; vertical-align: top;">完成代码填空与完整结果检查</td>
<td style="text-align: left; vertical-align: top;">30分钟</td>
</tr>
</tbody>
</table>

<small>时间说明：建议时间用于阅读与讲解，不包含自行运行程序和独立完成练习的时间。</small>

<small>运行提示：线上环境不提供昇腾950（A5）。各节已保留代码和实测结果，可以直接阅读；需要运行时，请自行准备昇腾950环境。</small>

## 小结

第3章介绍 Vector 计算，第4章介绍 Cube 矩阵乘。本章以 Matmul + Bias 为例，把两者接成一条完整的计算链：C 核计算矩阵乘，V核加偏置。

我们先实现固定形状的版本，再让它处理行数不同的输入。下一节从固定形状的融合算子开始。

## 课后练习

1. 在 Matmul + Bias 中，哪些计算主要由 C 核完成，哪些主要由 V 核完成？得到矩阵乘结果 X 后，整条计算链是否已经执行完毕？
2. 与前两章分别学习 Vector 算子和 GEMM 算子相比，本章还需要解决什么问题？
3. 固定输入形状的融合算子写好后，5.3将继续解决什么问题？

<details>
<summary>展开练习解析</summary>

1. 矩阵乘主要由 **Cube（C核）** 完成，偏置相加主要由 **Vector（V核）** 完成。得到 `X` 后，还要加偏置才能得到 `Y`；完整程序还包括数据搬运和结果写回。
2. 本章进一步学习怎样在同一个算子中衔接矩阵计算和向量计算。除了写出各步计算，还要安排好每一步需要的数据，以及C核与V核之间的结果传递。
3. 5.3讨论怎样处理行数不同的输入：先明确支持哪些行数，再在调用前检查输入，拒绝不符合要求的情况。

</details>

[章节导航](README.md) · [下一节](05.02_matmul_bias.ipynb)